In [ ]:
import pickle, os, json
import numpy as np
import casadi as cd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from narx_functions import create_narx_trajectory, Hyperparameters
from keras.layers import Dense, Input, Concatenate, Add
from keras.optimizers import Adam, AdamW
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras import Model
from keras import regularizers

from collections import deque
from matplotlib import pyplot as plt

plt.rcParams.update({
    "font.size": 12,
    # "text.usetex": True,
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.serif": ["Roman"],
})

from datetime import datetime
timestamp = datetime.now().strftime("%y%m%d_%H%M%S")

In [ ]:
hyperparameters = Hyperparameters()

hyperparameters.total_number_of_steps  = 1
hyperparameters.n_dense_layers = 1
hyperparameters.n_neurons_per_layer = 64
hyperparameters.learning_rate = 1e-4
hyperparameters.batch_size = 2048
hyperparameters.train_test_split_ratio = 0.1
hyperparameters.n_epochs = 200
hyperparameters.loss = "mse"
hyperparameters.regularization_l2 = 1e-4
# LI, WI, W_L_D_c1, w_L_B_c1, T_tr9
hyperparameters.noise_on_states = np.array([0.1, 10, 0.005, 0.005, 0.2])  # Standard deviation of Gaussian noise to add to states
hyperparameters.n_augmentation = 10

hyperparameters.t_step_of_unmodified_data = 120.0
hyperparameters.t_step_for_model = 120.0


In [ ]:
performance_metrics = {}

In [ ]:
# data_path = os.path.join("data", "LVcolumn_Sampling", "260506_141325_Sampling_1000.pkl")
data_path = os.path.join("data", "LVcolumn_Sampling", f"260618_092315_Sampling_10.pkl")
hyperparameters.data_path = data_path

with open(data_path, "rb") as f:
    full_data = pickle.load(f)

In [ ]:
# testdata_path = os.path.join("data", "LVcolumn_Sampling", "260507_111741_Sampling_100_test.pkl")
testdata_path = os.path.join("data", "LVcolumn_Sampling", f"260618_092341_Sampling_10_test.pkl")

with open(testdata_path, "rb") as f:
    full_test_data = pickle.load(f)

In [ ]:
skipping_factor = int(hyperparameters.t_step_for_model // hyperparameters.t_step_of_unmodified_data)
print("Skipping factor:", skipping_factor)

skipping_rest = hyperparameters.t_step_for_model % hyperparameters.t_step_of_unmodified_data
if skipping_rest != 0:
    print("Warning: skipping rest is not zero:", skipping_rest)



In [ ]:
model_path = os.path.basename(hyperparameters.data_path).split(".pkl")[0]
hyperparameters.model_path = os.path.join("surr_models", model_path, f"{timestamp}_narx_{hyperparameters.total_number_of_steps}step")
hyperparameters.make_dirs(overwrite=False)

In [ ]:
full_data_x = full_data["x"]
full_data_u = full_data["u"]

full_data_x_cleaned = []
for df in full_data_x:
    df_cleaned = df.copy()
    df_cleaned = df_cleaned[["x_e0_LI", "x_e0_WI", "x_e0_w_L_D_c1", "x_e0_w_L_B_c1", "x_e0_T_tr9"]]
    full_data_x_cleaned.append(df_cleaned)
full_data_x = full_data_x_cleaned.copy()

print("State variables used for training:", full_data_x[0].columns.tolist())

In [ ]:
full_test_data_x = full_test_data["x"]
full_test_data_u = full_test_data["u"]

full_test_data_x_cleaned = []
for df in full_test_data_x:
    df_cleaned = df.copy()
    df_cleaned = df_cleaned[["x_e0_LI", "x_e0_WI", "x_e0_w_L_D_c1", "x_e0_w_L_B_c1", "x_e0_T_tr9"]]
    full_test_data_x_cleaned.append(df_cleaned)
full_test_data_x = full_test_data_x_cleaned.copy()

print("State variables used for testing:", full_test_data_x[0].columns.tolist())

In [ ]:
full_data_u_cleaned = []
full_data_x_cleaned = []

for dfx, df_u in zip(full_data_x, full_data_u):
    reduced_u_data = df_u.loc[df_u["u_dummy_rr"] < 1.0]
    reduced_x_data = dfx.loc[reduced_u_data.index]

    if len(reduced_u_data) < hyperparameters.total_number_of_steps + 1:
        continue
    
    full_data_x_cleaned.append(reduced_x_data.reset_index().drop("index", axis=1))
    full_data_u_cleaned.append(reduced_u_data.reset_index().drop("index", axis=1))

full_data_x = full_data_x_cleaned.copy()
full_data_u = full_data_u_cleaned.copy()

print("Number of trajectories after cleaning:", len(full_data_x))

In [ ]:
full_test_data_u_cleaned = []
full_test_data_x_cleaned = []

for dfx, df_u in zip(full_test_data_x, full_test_data_u):
    reduced_u_data = df_u.loc[df_u["u_dummy_rr"] < 1.0]
    reduced_x_data = dfx.loc[reduced_u_data.index]

    if len(reduced_u_data) < hyperparameters.total_number_of_steps + 1:
        continue
    
    full_test_data_x_cleaned.append(reduced_x_data.reset_index().drop("index", axis=1))
    full_test_data_u_cleaned.append(reduced_u_data.reset_index().drop("index", axis=1))

full_test_data_x = full_test_data_x_cleaned.copy()
full_test_data_u = full_test_data_u_cleaned.copy()

print("Number of test trajectories after cleaning:", len(full_test_data_x))

In [ ]:
skipping_factor = int(hyperparameters.t_step_for_model // hyperparameters.t_step_of_unmodified_data)
print("Skipping factor:", skipping_factor)

skipping_rest = hyperparameters.t_step_for_model % hyperparameters.t_step_of_unmodified_data
if skipping_rest != 0:
    print("Warning: skipping rest is not zero:", skipping_rest)

full_data_u_cleaned = []
full_data_x_cleaned = []

for dfx, dfu in zip(full_data_x, full_data_u):

    full_data_x_cleaned.append(dfx[0::skipping_factor].reset_index().drop("index", axis=1))
    full_data_u_cleaned.append(dfu[0::skipping_factor].reset_index().drop("index", axis=1))

full_data_x = full_data_x_cleaned.copy()
full_data_u = full_data_u_cleaned.copy()

print(f"Only used every {skipping_factor}-th data point for training. This is equivalent to a time step of {hyperparameters.t_step_for_model} seconds instead of {hyperparameters.t_step_of_unmodified_data}.")

In [ ]:
full_test_data_u_cleaned = []
full_test_data_x_cleaned = []

for dfx, dfu in zip(full_test_data_x, full_test_data_u):

    full_test_data_x_cleaned.append(dfx[0::skipping_factor].reset_index().drop("index", axis=1))
    full_test_data_u_cleaned.append(dfu[0::skipping_factor].reset_index().drop("index", axis=1))

full_test_data_x = full_test_data_x_cleaned.copy()
full_test_data_u = full_test_data_u_cleaned.copy()

print(f"Only used every {skipping_factor}-th data point for testing. This is equivalent to a time step of {hyperparameters.t_step_for_model} seconds instead of {hyperparameters.t_step_of_unmodified_data}.")

In [ ]:
# Noisification
full_data_u_noisy = []
full_data_x_noisy = []
for idx in range(hyperparameters.n_augmentation):
    for dfx, dfu in zip(full_data_x, full_data_u):
        noise = np.random.normal(0, hyperparameters.noise_on_states, size=dfx.shape)
        dfx_noisy = dfx + noise
        full_data_x_noisy.append(dfx_noisy)
        full_data_u_noisy.append(dfu)

full_data_x = full_data_x_noisy.copy()
full_data_u = full_data_u_noisy.copy()

print("Total number of trajectories after noisification:", len(full_data_x))

In [ ]:
# Noisification
full_test_data_u_noisy = []
full_test_data_x_noisy = []
for idx in range(hyperparameters.n_augmentation):
    for dfx, dfu in zip(full_test_data_x, full_test_data_u):
        noise = np.random.normal(0, hyperparameters.noise_on_states, size=dfx.shape)
        dfx_noisy = dfx + noise
        full_test_data_x_noisy.append(dfx_noisy)
        full_test_data_u_noisy.append(dfu)

full_test_data_x = full_test_data_x_noisy.copy()
full_test_data_u = full_test_data_u_noisy.copy()

print("Total number of test trajectories after noisification:", len(full_test_data_x))

In [ ]:
hyperparameters.x_features = full_data_x[0].columns.tolist()
hyperparameters.u_features = full_data_u[0].columns.tolist()


hyperparameters.nx = full_data_x[0].shape[1]
hyperparameters.nu = full_data_u[0].shape[1]

xlabels = full_data_x[0].columns.tolist()
ulabels = full_data_u[0].columns.tolist()
all_labels = xlabels + ulabels

train_data_x, val_data_x = train_test_split(
    full_data_x, test_size=hyperparameters.train_test_split_ratio, random_state=hyperparameters.train_test_split_seed
)
train_data_u, val_data_u = train_test_split(
    full_data_u, test_size=hyperparameters.train_test_split_ratio, random_state=hyperparameters.train_test_split_seed
)

train_data = {"x": train_data_x, "u": train_data_u}
val_data = {"x": val_data_x, "u": val_data_u}
test_data = {"x": full_test_data_x, "u": full_test_data_u}


In [ ]:
performance_metrics["n_augmentations"] = hyperparameters.n_augmentation

performance_metrics["n_trajectories_train"] = len(train_data_x)
performance_metrics["n_trajectories_val"] = len(val_data_x)
performance_metrics["n_trajectories_test"] = len(full_test_data_x)
performance_metrics["n_trajectories_total"] = performance_metrics["n_trajectories_train"] + performance_metrics["n_trajectories_val"] + performance_metrics["n_trajectories_test"]

performance_metrics["n_samples_train"] = sum([len(df) for df in train_data_x])
performance_metrics["n_samples_val"] = sum([len(df) for df in val_data_x])
performance_metrics["n_samples_test"] = sum([len(df) for df in full_test_data_x])
performance_metrics["n_samples_total"] = performance_metrics["n_samples_train"] + performance_metrics["n_samples_val"] + performance_metrics["n_samples_test"]

print(performance_metrics)

In [ ]:
# Scaling
scaler_x = MinMaxScaler()
scaler_u = MinMaxScaler()

# Fit scalers on training data
train_data_x_concat = np.concatenate(train_data_x, axis=0)
train_data_u_concat = np.concatenate(train_data_u, axis=0)

scaler_x.fit(train_data_x_concat)
scaler_u.fit(train_data_u_concat)

with open(os.path.join(hyperparameters.model_path, "scaler_x.pkl"), "wb") as f:
    pickle.dump(scaler_x, f)

with open(os.path.join(hyperparameters.model_path, "scaler_u.pkl"), "wb") as f:
    pickle.dump(scaler_u, f)

# Transform both training and validation data
train_data_x_scaled = [scaler_x.transform(x) for x in train_data_x]
train_data_u_scaled = [scaler_u.transform(u) for u in train_data_u]

val_data_x_scaled = [scaler_x.transform(x) for x in val_data_x]
val_data_u_scaled = [scaler_u.transform(u) for u in val_data_u]

test_data_x_scaled = [scaler_x.transform(x) for x in full_test_data_x]
test_data_u_scaled = [scaler_u.transform(u) for u in full_test_data_u]


In [ ]:
print("Ranges for state variables:")
for i, label in enumerate(hyperparameters.x_features):
    data_min = np.min(train_data_x_concat[:, i])
    data_max = np.max(train_data_x_concat[:, i])
    print(f"{label}: min = {data_min:.4f}, max = {data_max:.4f}")

print()
print("Ranges for input variables:")
for i, label in enumerate(hyperparameters.u_features):
    data_min = np.min(train_data_u_concat[:, i])
    data_max = np.max(train_data_u_concat[:, i])
    print(f"{label}: min = {data_min:.4f}, max = {data_max:.4f}")

In [ ]:
all_data_scaled_stacked = [np.hstack((item1, item2)) for item1, item2 in zip(train_data_x_scaled, train_data_u_scaled)]
all_data_scaled_stacked = np.vstack(all_data_scaled_stacked)

In [ ]:
ncols = 3
nrows = 3
scaling_factor = 0.6
figsize = (ncols * 4 * scaling_factor, nrows * 3 * scaling_factor)
fig, ax = plt.subplots(ncols = ncols, nrows = nrows, figsize=figsize, constrained_layout=True)

for idx, axis in enumerate(ax.flatten()):
    if idx < all_data_scaled_stacked.shape[1]:
        axis.hist(all_data_scaled_stacked[:, idx], bins=50, density=False)
        axis.set_title(all_labels[idx])

plt.suptitle("Histograms of scaled training data features")

plt.savefig(os.path.join(hyperparameters.model_path, "figs", "training_data_feature_histograms.png"))

In [ ]:
all_data_stacked = [np.hstack((item1, item2)) for item1, item2 in zip(train_data_x, train_data_u)]
all_data_stacked = np.vstack(all_data_stacked)

In [ ]:
ncols = 3
nrows = 3
scaling_factor = 0.6
figsize = (ncols * 4 * scaling_factor, nrows * 3 * scaling_factor)
fig, ax = plt.subplots(ncols = ncols, nrows = nrows, figsize=figsize, constrained_layout=True)

for idx, axis in enumerate(ax.flatten()):
    if idx < all_data_stacked.shape[1]:
        axis.hist(all_data_stacked[:, idx], bins=50, density=False)
        axis.set_title(all_labels[idx])

plt.suptitle("Histograms of training data features")

plt.savefig(os.path.join(hyperparameters.model_path, "figs", "training_data_unscaled_feature_histograms.png"))

In [ ]:
train_narx_input_x_data_scaled = []
train_narx_input_u_data_scaled = []
train_narx_output_data_scaled = []

for idx, (trajectory_x, trajectory_u) in enumerate(zip(train_data_x_scaled, train_data_u_scaled)):
    narx_input_x, narx_input_u, narx_output = create_narx_trajectory(
        trajectory_x,
        trajectory_u,
        total_number_of_steps=hyperparameters.total_number_of_steps,
    )

    train_narx_input_x_data_scaled.append(narx_input_x)
    train_narx_input_u_data_scaled.append(narx_input_u)
    train_narx_output_data_scaled.append(narx_output)

val_narx_input_x_data_scaled = []
val_narx_input_u_data_scaled = []
val_narx_output_data_scaled = []

for idx, (trajectory_x, trajectory_u) in enumerate(zip(val_data_x_scaled, val_data_u_scaled)):
    narx_input_x, narx_input_u, narx_output = create_narx_trajectory(
        trajectory_x,
        trajectory_u,
        total_number_of_steps=hyperparameters.total_number_of_steps,
    )

    val_narx_input_x_data_scaled.append(narx_input_x)
    val_narx_input_u_data_scaled.append(narx_input_u)
    val_narx_output_data_scaled.append(narx_output)

test_narx_input_x_data_scaled = []
test_narx_input_u_data_scaled = []
test_narx_output_data_scaled = []

for idx, (trajectory_x, trajectory_u) in enumerate(zip(test_data_x_scaled, test_data_u_scaled)):
    narx_input_x, narx_input_u, narx_output = create_narx_trajectory(
        trajectory_x,
        trajectory_u,
        total_number_of_steps=hyperparameters.total_number_of_steps,
    )
    
    test_narx_input_x_data_scaled.append(narx_input_x)
    test_narx_input_u_data_scaled.append(narx_input_u)
    test_narx_output_data_scaled.append(narx_output)

In [ ]:
train_narx_input_x_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in train_narx_input_x_data_scaled:
        train_narx_input_x_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nx,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    train_narx_input_x_data_scaled_stacked[idx] = np.vstack(
        train_narx_input_x_data_scaled_stacked[idx]
    )

train_narx_input_u_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in train_narx_input_u_data_scaled:
        train_narx_input_u_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nu,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    train_narx_input_u_data_scaled_stacked[idx] = np.vstack(
        train_narx_input_u_data_scaled_stacked[idx]
    )

train_narx_output_data_scaled = np.vstack(train_narx_output_data_scaled)

In [ ]:
val_narx_input_x_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in val_narx_input_x_data_scaled:
        val_narx_input_x_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nx,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    val_narx_input_x_data_scaled_stacked[idx] = np.vstack(
        val_narx_input_x_data_scaled_stacked[idx]
    )

val_narx_input_u_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in val_narx_input_u_data_scaled:
        val_narx_input_u_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nu,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    val_narx_input_u_data_scaled_stacked[idx] = np.vstack(
        val_narx_input_u_data_scaled_stacked[idx]
    )

val_narx_output_data_scaled = np.vstack(val_narx_output_data_scaled)

In [ ]:
test_narx_input_x_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in test_narx_input_x_data_scaled:
        test_narx_input_x_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nx,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    test_narx_input_x_data_scaled_stacked[idx] = np.vstack(
        test_narx_input_x_data_scaled_stacked[idx]
    )

test_narx_input_u_data_scaled_stacked = [[] for _ in range(hyperparameters.total_number_of_steps)]
for idx in range(hyperparameters.total_number_of_steps):
    for traj in test_narx_input_u_data_scaled:
        test_narx_input_u_data_scaled_stacked[idx].append(
            traj[idx] if idx < len(traj) else np.zeros((hyperparameters.nu,))
        )

for idx in range(hyperparameters.total_number_of_steps):
    test_narx_input_u_data_scaled_stacked[idx] = np.vstack(
        test_narx_input_u_data_scaled_stacked[idx]
    )

test_narx_output_data_scaled = np.vstack(test_narx_output_data_scaled)

In [ ]:

input_x_layers = []
input_u_layers = []

for idx in range(hyperparameters.total_number_of_steps):
    input_x_layers.append(Input(shape=(train_data_x[0].shape[1],), name=f"input_x_t-{hyperparameters.total_number_of_steps - idx}"))
    input_u_layers.append(Input(shape=(train_data_u[0].shape[1],), name=f"input_u_t-{hyperparameters.total_number_of_steps - idx}"))

stacked_x_inputs = Concatenate(name="stacked_x_inputs")(input_x_layers)
stacked_u_inputs = Concatenate(name="stacked_u_inputs")(input_u_layers)

stacked_inputs = Concatenate(name="stacked_inputs")([stacked_x_inputs, stacked_u_inputs])

for idx in range(hyperparameters.n_dense_layers):
    if idx == 0:
        dense_layer = Dense(hyperparameters.n_neurons_per_layer, activation="gelu", name=f"dense_{idx+1}", kernel_regularizer=regularizers.l2(hyperparameters.regularization_l2), bias_regularizer=regularizers.l2(hyperparameters.regularization_l2))(stacked_inputs)
    else:
        dense_layer = Dense(hyperparameters.n_neurons_per_layer, activation="gelu", name=f"dense_{idx+1}", kernel_regularizer=regularizers.l2(hyperparameters.regularization_l2), bias_regularizer=regularizers.l2(hyperparameters.regularization_l2))(dense_layer)

# delta_layer = Dense(nx, activation="linear", name="delta_layer")(dense_layer)
# output_layer = Add(name="output_layer")([input_x_layers[-1], delta_layer])

output_layer = Dense(hyperparameters.nx, activation="linear", name="output_layer")(dense_layer)

narx = Model(name = f"narx_model_{hyperparameters.total_number_of_steps}", inputs = input_x_layers + input_u_layers, outputs=output_layer)

narx.summary()

In [ ]:
narx.compile(
    optimizer = AdamW(learning_rate=hyperparameters.learning_rate),
    loss = hyperparameters.loss,
    metrics = ["mae"],
    run_eagerly=False
)

In [ ]:
history = narx.fit(
    x = train_narx_input_x_data_scaled_stacked + train_narx_input_u_data_scaled_stacked,
    y = train_narx_output_data_scaled,
    epochs = hyperparameters.n_epochs,
    batch_size = hyperparameters.batch_size,
    validation_data = (val_narx_input_x_data_scaled_stacked + val_narx_input_u_data_scaled_stacked, val_narx_output_data_scaled),
    callbacks = [
        ModelCheckpoint(
            filepath = os.path.join(hyperparameters.model_path, f"best_weights.weights.h5"),
            monitor = "val_loss",
            save_best_only = True,
            save_weights_only = True,
            verbose = 1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=100,
                verbose=1,
                min_lr=1e-7,
                min_delta=1e-7,
            ),
            ]
)

In [ ]:
with open(os.path.join(hyperparameters.model_path, "training_history.pkl"), "wb") as f:
    pickle.dump(history.history, f)
narx.save(filepath = os.path.join(hyperparameters.model_path, "narx_model.keras"))
hyperparameters.save()

In [ ]:
ncols = 2
fig, ax = plt.subplots(ncols = ncols, figsize=(ncols * 6, 4), constrained_layout=True)

ax[0].plot(history.history['loss'], label='Training Loss')
ax[0].plot(history.history['val_loss'], label='Validation Loss')

ax[1].plot(history.history['mae'], label='Training MAE')
ax[1].plot(history.history['val_mae'], label='Validation MAE')

for axis in ax:
    axis.grid(True)
    axis.legend(loc = "upper right")
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.set_yscale("log")
    # axis.set_xlim([-1, hyperparameters.n_epochs])

plt.suptitle("NARX Model Training History")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "surrogate_learning_curve_full.png"))


In [ ]:
ncols = 1
nrows = 1
scaling_factor = 1.0
figsize = (ncols * 5 * scaling_factor, 3 * scaling_factor)

fig, ax = plt.subplots(ncols = ncols, figsize=figsize, constrained_layout=True)
ax = [ax]

ax[0].plot(1 + np.arange(0, len(history.history['loss'])), history.history['loss'], label='Training')
ax[0].plot(1 + np.arange(0, len(history.history['val_loss'])), history.history['val_loss'], label='Validation')

for axis in ax:
    axis.grid(True)
    axis.legend(loc = "upper right")
    axis.set_xlabel("Epoch")
    axis.set_ylabel(r"$\mathrm{MSE}\,/\,-$")
    axis.set_yscale("log")
    axis.set_xlim([0, hyperparameters.n_epochs])

# plt.suptitle("NARX Model Training History")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "surrogate_learning_curve.png"), dpi = 1200.0)
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "surrogate_learning_curve.pdf"), dpi = 300.0)


In [ ]:
from transform_to_do_mpc import transform_keras_to_do_mpc_for_rl
from _rl_simulator import RL_Simulator as Simulator

do_mpc_model = transform_keras_to_do_mpc_for_rl(narx, scaler_x, scaler_u, hyperparameters)
do_mpc_model.setup()
with open(os.path.join(hyperparameters.model_path, "do_mpc_model_rl.pkl"), "wb") as f:
    pickle.dump(do_mpc_model, f)

simulator = Simulator(do_mpc_model)
simulator.settings.t_step = 30.0

rlp_template = simulator.get_rlp_template()
simulator.set_rlp_fun(lambda tnow: rlp_template)
simulator.setup()

In [ ]:
train_prediction = narx(train_narx_input_x_data_scaled_stacked + train_narx_input_u_data_scaled_stacked)

train_mse = np.mean((train_prediction - train_narx_output_data_scaled) ** 2)
performance_metrics["train_mse"] = float(np.float64(train_mse))
print(f"Train MSE: {train_mse:.6e}")

train_rmse = np.sqrt(train_mse)
performance_metrics["train_rmse"] = float(np.float64(train_rmse))
print(f"Train RMSE: {train_rmse:.6e}")

train_mae = np.mean(np.abs(train_prediction - train_narx_output_data_scaled))
performance_metrics["train_mae"] = float(np.float64(train_mae))
print(f"Train MAE: {train_mae:.6e}")

In [ ]:
ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 4 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

for idx, axis in enumerate(ax.flatten()):
    if idx < train_narx_output_data_scaled.shape[1]:
        axis.plot(train_narx_output_data_scaled[:, idx], train_prediction[:, idx], 'o', markersize=2)
        axis.set_xlabel("True Values")
        axis.set_ylabel("Predicted Values")
        axis.set_title(all_labels[idx])
        axis.plot([0, 1], [0, 1], 'k-')

plt.suptitle("NARX Model Predictions vs True Values (Training Data)")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "parity_plot_train_data.png"))

In [ ]:
val_prediction = narx(val_narx_input_x_data_scaled_stacked + val_narx_input_u_data_scaled_stacked)

val_mse = np.mean((val_prediction - val_narx_output_data_scaled) ** 2)
performance_metrics["val_mse"] = float(np.float64(val_mse))
print(f"Validation MSE: {val_mse:.6e}")

val_rmse = np.sqrt(val_mse)
performance_metrics["val_rmse"] = float(np.float64(val_rmse))
print(f"Validation RMSE: {val_rmse:.6e}")

val_mae = np.mean(np.abs(val_prediction - val_narx_output_data_scaled))
performance_metrics["val_mae"] = float(np.float64(val_mae))
print(f"Validation MAE: {val_mae:.6e}")

In [ ]:
ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 4 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

for idx, axis in enumerate(ax.flatten()):
    if idx < val_narx_output_data_scaled.shape[1]:
        axis.plot(val_narx_output_data_scaled[:, idx], val_prediction[:, idx], 'o', markersize=2)
        axis.set_xlabel("True Values")
        axis.set_ylabel("Predicted Values")
        axis.set_title(all_labels[idx])
        axis.plot([0, 1], [0, 1], 'k-')

plt.suptitle("NARX Model Predictions vs True Values (Validation Data)")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "parity_plot_val_data.png"))

In [ ]:
test_prediction = narx(test_narx_input_x_data_scaled_stacked + test_narx_input_u_data_scaled_stacked)

test_mse = np.mean((test_prediction - test_narx_output_data_scaled) ** 2)
performance_metrics["test_mse"] = float(np.float64(test_mse))
print(f"Test MSE: {test_mse:.6e}")

test_rmse = np.sqrt(test_mse)
performance_metrics["test_rmse"] = float(np.float64(test_rmse))
print(f"Test RMSE: {test_rmse:.6e}")

test_mae = np.mean(np.abs(test_prediction - test_narx_output_data_scaled))
performance_metrics["test_mae"] = float(np.float64(test_mae))
print(f"Test MAE: {test_mae:.6e}")

In [ ]:
ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 4 * scaling_factor)


fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

for idx, axis in enumerate(ax.flatten()):
    if idx < test_narx_output_data_scaled.shape[1]:
        axis.plot(test_narx_output_data_scaled[:, idx], test_prediction[:, idx], 'o', markersize=2)
        axis.set_xlabel("True Values")
        axis.set_ylabel("Predicted Values")
        axis.set_title(all_labels[idx])
        axis.plot([0, 1], [0, 1], 'k-')

plt.suptitle("NARX Model Predictions vs True Values (Test Data)")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "parity_plot_test_data.png"))

In [ ]:
start_at = 2

In [ ]:
simulator.reset_history()

test_trajectory_x = test_data_x_scaled[0]
test_trajectory_u = test_data_u_scaled[0]

test_trajectory_stacked = np.hstack((test_trajectory_x, test_trajectory_u))

narx_input_x_val_traj = deque(maxlen=hyperparameters.total_number_of_steps)
narx_input_u_val_traj = deque(maxlen=hyperparameters.total_number_of_steps)

predicted_trajectory = []

for t in range(start_at, start_at + hyperparameters.total_number_of_steps):
    narx_input_x_val_traj.append(test_trajectory_x[t].reshape(1, -1))
    narx_input_u_val_traj.append(test_trajectory_u[t].reshape(1, -1))

    predicted_trajectory.append(test_trajectory_x[t])

do_mpc_input_x = []
do_mpc_input_u = []

predicted_trajectory_do_mpc = []
for item_x, item_u in zip(narx_input_x_val_traj, narx_input_u_val_traj):
    do_mpc_input_x.append(scaler_x.inverse_transform(item_x).T)
    do_mpc_input_u.append(scaler_u.inverse_transform(item_u).T)

    predicted_trajectory_do_mpc.append(scaler_x.inverse_transform(item_x).reshape(-1))

simulator.x0.master = cd.DM(np.vstack([*do_mpc_input_x, *do_mpc_input_u[:-1]]))

for t in range(start_at + hyperparameters.total_number_of_steps, test_trajectory_x.shape[0]):

    prediction = narx(list(narx_input_x_val_traj) + list(narx_input_u_val_traj))
    predicted_trajectory.append(prediction.numpy().reshape(-1))

    narx_input_x_val_traj.append(prediction.numpy().reshape(1, -1))
    narx_input_u_val_traj.append(test_trajectory_u[t].reshape(1, -1))

    do_mpc_state_next = simulator.make_step(u0 = do_mpc_input_u[-1])
    do_mpc_state_next = simulator.sim_x(do_mpc_state_next)["_x", "x_t-1"].full()

    do_mpc_input_u.append(scaler_u.inverse_transform(test_trajectory_u[t].reshape(1, -1)).T)
    predicted_trajectory_do_mpc.append(do_mpc_state_next.flatten())

predicted_trajectory = np.vstack(predicted_trajectory)
predicted_trajectory_do_mpc = np.vstack(predicted_trajectory_do_mpc)
predicted_trajectory_do_mpc_scaled = scaler_x.transform(predicted_trajectory_do_mpc)

In [ ]:
# Time vectors (minutes)
time_true = np.arange(test_trajectory_stacked.shape[0]) * hyperparameters.t_step_for_model / 60.0
time_pred = (start_at + np.arange(predicted_trajectory.shape[0])) * hyperparameters.t_step_for_model / 60.0

# Split scaled stacked trajectory into x/u, then unscale
test_x_scaled = test_trajectory_stacked[:, :hyperparameters.nx]
test_u_scaled = test_trajectory_stacked[:, hyperparameters.nx:]

test_x_unscaled = scaler_x.inverse_transform(test_x_scaled)
test_u_unscaled = scaler_u.inverse_transform(test_u_scaled)
test_trajectory_unscaled = np.hstack((test_x_unscaled, test_u_unscaled))

# Unscale NARX state predictions
predicted_trajectory_unscaled = scaler_x.inverse_transform(predicted_trajectory)

ncols = 3
nrow = 3
scaling_factor = 0.5
figsize = (ncols*5 * scaling_factor, 4*nrow * scaling_factor)

fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True, sharex = True)

for idx, axis in enumerate(ax.flatten()):
    if idx >= test_trajectory_unscaled.shape[1]:
        continue

    
    if idx < hyperparameters.nx:
        axis.plot(time_true, test_trajectory_unscaled[:, idx], label="True", linewidth=2)
        axis.plot(time_pred, predicted_trajectory_unscaled[:, idx], label="Predicted", linewidth=2, linestyle="--")
    else:
        axis.step(time_true, test_trajectory_unscaled[:, idx], label="True", linewidth=2, where="post")

    axis.title.set_text(all_labels[idx])
    axis.set_ylabel("Unscaled Values")
    axis.legend(loc="upper right")
    axis.grid()

ax[0, 0].set_ylabel(r"LI Sumpf / \%")
ax[0, 1].set_ylabel(r"WI Product / g")
ax[0, 2].set_ylabel(r"QI Product / $-$")
ax[1, 0].set_ylabel(r"QI Sumpf / $-$")
ax[1, 1].set_ylabel(r"TI Sumpf / K")
ax[1, 2].set_ylabel(r"rr / -")
ax[2, 0].set_ylabel(r"$\dot{Q}$ / kW")


ax[-1, 0].set_xlabel("Time / min")
ax[-1, 1].set_xlabel("Time / min")
ax[-1, 2].set_xlabel("Time / min")

ax[0, 0].set_xlim([time_true[0], time_true[-1]])

plt.suptitle("NARX Model Multi-step Prediction on test Trajectory (Unscaled)")
plt.savefig(os.path.join(hyperparameters.model_path, "figs", "multi_step_prediction_test_trajectory_unscaled.png"))

In [ ]:
with open(os.path.join(hyperparameters.model_path, "performance_metrics.json"), "w") as f:
    json.dump(performance_metrics, f, indent=4)

In [ ]:
alpha = 0.2
percentage_of_ylim = 0.2

ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)

lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])

ubx[1] = 11e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 6.0])



plt.close("all")

fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

y_data_experiment = test_x_unscaled
u_data_experiment = test_u_unscaled
time_data_experiment = time_true

time_data_experiment_plot = time_data_experiment


# Plot the experiments
label = "Test data"
ax[0, 0].plot(time_data_experiment_plot, y_data_experiment[:, 1] * 1e-3, marker = "None", linestyle = "solid", label = label)
ax[0, 1].plot(time_data_experiment_plot, y_data_experiment[:, 2], marker = "None", linestyle = "solid")
ax[1, 0].plot(time_data_experiment_plot, y_data_experiment[:, 3], marker = "None", linestyle = "solid")
ax[1, 1].plot(time_data_experiment_plot, y_data_experiment[:, 4] - 273.15, marker = "None", linestyle = "solid")
ax[2, 0].step(time_data_experiment_plot, u_data_experiment[:, 0], where="post")
ax[2, 1].step(time_data_experiment_plot, u_data_experiment[:, 1], where="post")

t_min = time_data_experiment_plot[0]
t_max = time_data_experiment_plot[-1] + 2


# Plot the simulation with the best probe parameters
y_data_simulation = predicted_trajectory_unscaled
u_data_simulation = test_u_unscaled
time_data_simulation = time_pred

time_data_simulation_plot = time_data_simulation

label = f"Surrogate"
ax[0, 0].plot(time_data_simulation_plot, y_data_simulation[:, 1] * 1e-3, label = label, linestyle = 'dashed')
ax[0, 1].plot(time_data_simulation_plot, y_data_simulation[:, 2], linestyle = 'dashed')
ax[1, 0].plot(time_data_simulation_plot, y_data_simulation[:, 3], linestyle = 'dashed')
ax[1, 1].plot(time_data_simulation_plot, y_data_simulation[:, 4] - 273.15, linestyle = 'dashed')



ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
ax[0, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
ax[0, 0].set_yticks(np.arange(0.0, 12.6, step=3))


ax[0, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")


ax[1, 0].set_ylabel(r"$w_\mathrm{B}~/~-$")
ax[1, 0].fill_between(np.array([t_min, t_max]).flatten(), lbx[3], lbx[3] - (ubx[3] - lbx[3]), color="tab:red", alpha=alpha, label = r"Constraints")
ax[1, 0].set_ylim([lbx[3] - percentage_of_ylim * (ubx[3] - lbx[3]), 0.23])
ax[1, 0].set_yticks(np.arange(0.0, 0.231, step=0.07))


ax[1, 1].set_ylabel(r"$T_\mathrm{B}~/~^{\circ}$C")
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), lbx[4] - 273.15, lbx[4] - 273.15 - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].fill_between(np.array([t_min, t_max]).flatten(), 100.0, 100.0 + (100.0 + 273.15 - lbx[4]), color="tab:red", alpha=alpha)
ax[1, 1].set_ylim([lbx[4] - 273.15 - percentage_of_ylim * (100.0 + 273.15 - lbx[4]), 100.0 + percentage_of_ylim * (100.0 + 273.15 - lbx[4])])


ax[2, 0].set_ylabel(r"$rr^\mathrm{SP}~/~-$")
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), lbu[0], lbu[0] - (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].fill_between(np.array([t_min, t_max]).flatten(), ubu[0], ubu[0] + (ubu[0] - lbu[0]), color="tab:red", alpha=alpha)
ax[2, 0].set_ylim([lbu[0] - percentage_of_ylim * (ubu[0] - lbu[0]), ubu[0] + percentage_of_ylim * (ubu[0] - lbu[0])])
ax[2, 0].set_yticks(np.arange(0.7, 1.01, step=0.1))


ax[2, 1].set_ylabel(r"$\dot{Q}~/~$kW")
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), lbu[1], lbu[1] - (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].fill_between(np.array([t_min, t_max]).flatten(), ubu[1], ubu[1] + (ubu[1] - lbu[1]), color="tab:red", alpha=alpha)
ax[2, 1].set_ylim([lbu[1] - percentage_of_ylim * (ubu[1] - lbu[1]), ubu[1] + percentage_of_ylim * (ubu[1] - lbu[1])])
ax[2, 1].set_yticks(np.arange(3.0, 6.1, step=1.0))


ax[-1, 0].set_xlabel(r"Time $t~/~$min")
ax[-1, 1].set_xlabel(r"Time $t~/~$min")


for axis in ax.flatten():
    axis.grid(True)
    axis.set_xlim([t_min, t_max])
    axis.set_xticks(np.arange(0, t_max+1, step=30))

fig.legend(loc = "outside lower center", ncol=3)

figpath = os.path.join(hyperparameters.model_path, "figs")
os.makedirs(figpath, exist_ok=True)
fig.savefig(os.path.join(figpath, f"fit_multistep_prediction_present.png"), dpi=1200)
fig.savefig(os.path.join(figpath, f"fit_multistep_prediction_present.pdf"), dpi=300)